# Requirement 2: Explicit plaintext boundary

> Only the trusted calling application and the approved execution environment may handle sensitive plaintext.

**Covered by:** Both

**How it is fulfilled:** The orchestrator encrypts data client side with a per-session key and uploads ciphertext. Plaintext exists only in the orchestrator and in the session's own sandbox after the key is installed there.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
sandbox, session_id = new_session()
print("session", session_id)

session fbf501f6


In [3]:
session_key = install_session_key(sandbox, session_id)
plaintext = b'{"client": "Acme", "matter": "Project Falcon", "amount": 1200}'
nonce = os.urandom(12)
ciphertext = nonce + AESGCM(session_key).encrypt(nonce, plaintext, session_id.encode())
sandbox.write_file("/tmp/input.enc", ciphertext)
stored = sandbox.read_file("/tmp/input.enc")
check("what the platform stores is ciphertext, not plaintext", b"Falcon" not in stored and stored == ciphertext)
check("the plaintext never appears in the sandbox config or labels", b"Falcon" not in str(sandbox.get()).encode())

PASS  what the platform stores is ciphertext, not plaintext
PASS  the plaintext never appears in the sandbox config or labels


In [4]:
cleanup(sandbox)
check("session deleted", not list(orchestrator().list_sandboxes(labels={"session": session_id})))

PASS  session deleted


**Caveat:** Anyone with Data Owner on the group can still read the sandbox, including keys and plaintext in memory. The plaintext boundary is enforced by keeping that role to the orchestrator.